# Mohamed Soueidatt - C12989

#### Solutions des exercices du chapitre 8

### Ex. 1 - `RationalNumber.simplify`

In [1]:
from math import gcd

class RationalNumber:
    def __init__(self, numerator, denominator=1):
        if denominator == 0:
            raise ZeroDivisionError('denominator must be non-zero')
        g = gcd(numerator, denominator)
        if denominator < 0:
            g = -g
        self.numerator = numerator // g
        self.denominator = denominator // g

    def simplify(self):
        return self.numerator, self.denominator

    def __repr__(self):
        return f"{self.numerator}/{self.denominator}"

q = RationalNumber(10, 15)
q.simplify()

(2, 3)

### Ex. 2 - Classe `Interval` et arithmetique des intervalles

In [2]:
class Interval:
    def __init__(self, left, right=None):
        if right is None:
            right = left
        if left > right:
            left, right = right, left
        self.left = float(left)
        self.right = float(right)

    @staticmethod
    def as_interval(value):
        return value if isinstance(value, Interval) else Interval(value, value)

    def __repr__(self):
        def fmt(x):
            return str(int(x)) if x.is_integer() else f"{x:.6g}"
        return f"[{fmt(self.left)}, {fmt(self.right)}]"

    def __contains__(self, x):
        return self.left <= x <= self.right

    def __add__(self, other):
        other = Interval.as_interval(other)
        return Interval(self.left + other.left, self.right + other.right)
    __radd__ = __add__

    def __sub__(self, other):
        other = Interval.as_interval(other)
        return Interval(self.left - other.right, self.right - other.left)

    def __rsub__(self, other):
        return Interval.as_interval(other) - self

    def __mul__(self, other):
        other = Interval.as_interval(other)
        products = [self.left*other.left, self.left*other.right,
                    self.right*other.left, self.right*other.right]
        return Interval(min(products), max(products))
    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Interval.as_interval(other)
        if 0 in other:
            raise ZeroDivisionError('division by an interval containing 0')
        return self * Interval(1/other.right, 1/other.left)

    def __rtruediv__(self, other):
        return Interval.as_interval(other) / self

    def __pow__(self, n):
        if not isinstance(n, int) or n < 1:
            raise ValueError('power must be a positive integer')
        values = [self.left**n, self.right**n]
        if n % 2 == 0 and self.left <= 0 <= self.right:
            return Interval(0, max(values))
        return Interval(min(values), max(values))

I = Interval(0, 0.1)
f = lambda x: 25*x**2 - 4*x + 1
(I + 2, 2 + I, I * 3, 3 * I, I**2, 0.05 in I, f(I))

([2, 2.1], [2, 2.1], [0, 0.3], [0, 0.3], [0, 0.01], True, [0.6, 1.25])

### Ex. 3 - Decorateur qui compte les appels

In [3]:
class CountCalls:
    instances = {}

    def __init__(self, func):
        self.func = func
        self.calls = 0
        CountCalls.instances[func.__name__] = self

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls
def square(x):
    return x*x

(square(3), square(4), square.calls)

(9, 16, 2)

### Ex. 4 - Erreur dans une mauvaise implementation de `__radd__`

In [4]:
class BadRationalNumber(RationalNumber):
    def __radd__(self, other):
        return other + self

try:
    5 + BadRationalNumber(10, 15)
except RecursionError as err:
    result = type(err).__name__

result

'RecursionError'

### Ex. 5 - Methode `reset` pour `CountCalls`

In [5]:
class CountCalls:
    instances = {}

    def __init__(self, func):
        self.func = func
        self.calls = 0
        CountCalls.instances[func.__name__] = self

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

    @classmethod
    def reset(cls):
        for counter in cls.instances.values():
            counter.calls = 0

@CountCalls
def f(x): return x + 1

@CountCalls
def g(x): return 2*x

f(1); f(2); g(3)
before = {name: obj.calls for name, obj in CountCalls.instances.items()}
CountCalls.reset()
after = {name: obj.calls for name, obj in CountCalls.instances.items()}
explanation = 'Replacing instances by {} would lose the registered decorated functions.'
before, after, explanation

({'f': 2, 'g': 1}, {'f': 0, 'g': 0}, 'Replacing instances by {} would lose the registered decorated functions.')